In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import sqlite3

# API for the Vancouver building permit dataset. Found from the Open Data website.
url = (
    "https://opendata.vancouver.ca/api/explore/v2.1/"
    "catalog/datasets/issued-building-permits/exports/csv"
)

response = requests.get(url)
response.raise_for_status()

# Turn API data into Pandas dataframe for cleaning and analysis.
df = pd.read_csv(StringIO(response.text), sep=";")

print(df.shape)
print(df.columns.tolist())

(51893, 20)
['permitnumber', 'permitnumbercreateddate', 'issuedate', 'permitelapseddays', 'projectvalue', 'typeofwork', 'address', 'projectdescription', 'permitcategory', 'applicant', 'applicantaddress', 'propertyuse', 'specificusecategory', 'buildingcontractor', 'buildingcontractoraddress', 'issueyear', 'geolocalarea', 'geom', 'yearmonth', 'geo_point_2d']


In [2]:
df = df.rename(columns={
    "permitnumber": "permit_number",
    "permitnumbercreateddate": "permit_number_created_date",
    "issuedate": "issue_date",
    "permitelapseddays": "permit_elapsed_days",
    "projectvalue": "project_value",
    "typeofwork": "type_of_work",
    "address": "address",
    "projectdescription": "project_description",
    "permitcategory": "permit_category",
    "applicant": "applicant",
    "applicantaddress": "applicant_address",
    "propertyuse": "property_use",
    "specificusecategory": "specific_use_category",
    "buildingcontractor": "building_contractor",
    "buildingcontractoraddress": "building_contractor_address",
    "issueyear": "issue_year",
    "geolocalarea": "geo_local_area",
    "geom": "geom",
    "yearmonth": "year_month",
    "geo_point_2d": "geo_point_2d"
})

In [3]:
print(df.columns.tolist())
print(df.shape)

['permit_number', 'permit_number_created_date', 'issue_date', 'permit_elapsed_days', 'project_value', 'type_of_work', 'address', 'project_description', 'permit_category', 'applicant', 'applicant_address', 'property_use', 'specific_use_category', 'building_contractor', 'building_contractor_address', 'issue_year', 'geo_local_area', 'geom', 'year_month', 'geo_point_2d']
(51893, 20)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51893 entries, 0 to 51892
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   permit_number                51893 non-null  str    
 1   permit_number_created_date   51893 non-null  str    
 2   issue_date                   51893 non-null  str    
 3   permit_elapsed_days          51893 non-null  int64  
 4   project_value                51893 non-null  float64
 5   type_of_work                 51893 non-null  str    
 6   address                      51680 non-null  str    
 7   project_description          51893 non-null  str    
 8   permit_category              29769 non-null  str    
 9   applicant                    51893 non-null  str    
 10  applicant_address            51717 non-null  str    
 11  property_use                 51887 non-null  str    
 12  specific_use_category        51885 non-null  str    
 13  building_contractor        

In [5]:
# Convert numeric fields from strings to int and float.
df['permit_elapsed_days'] = pd.to_numeric(
    df['permit_elapsed_days'],
    errors='coerce'
)

df['project_value'] = pd.to_numeric(
    df['project_value'],
    errors='coerce'
)

df['issue_year'] = pd.to_numeric(
    df['issue_year'],
    errors='coerce'
)

print(df[['permit_elapsed_days', 'project_value', 'issue_year']].dtypes)

permit_elapsed_days      int64
project_value          float64
issue_year               int64
dtype: object


In [6]:
# Remove records without coordinates.
df = df.dropna(subset=['geo_point_2d'])

# Split coordinates into latitude and longitude.
df[['latitude', 'longitude']] = (
    df['geo_point_2d']
    .str.split(',', expand=True)
)

df['latitude'] = pd.to_numeric(
    df['latitude'].str.strip(),
    errors='coerce'
)

df['longitude'] = pd.to_numeric(
    df['longitude'].str.strip(),
    errors='coerce'
)

# Remove invalid coordinates that do not match Vancouver coordinates (due to error).
df = df.dropna(subset=['latitude', 'longitude'])

df = df[
    df['latitude'].between(49.0, 49.5) &
    df['longitude'].between(-123.5, -122.5)
].copy()

# The geo_point_2d column is not longer necessary.
df = df.drop(columns=['geo_point_2d'])

print(df[['latitude', 'longitude']].head())

    latitude   longitude
0  49.263196 -123.039966
1  49.263196 -123.039966
2  49.285992 -123.118965
3  49.253087 -123.177011
4  49.265065 -123.112607


In [7]:
# Final data check.

print("Final dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate permit numbers:")
print(df['permit_number'].duplicated().sum())

print("\nProject value summary:")
print(df['project_value'].describe())

print("\nPermit elapsed days summary:")
print(df['permit_elapsed_days'].describe())

print("\nSample:")
display(df.head())

Final dataset shape: (51533, 21)

Data types:
permit_number                      str
permit_number_created_date         str
issue_date                         str
permit_elapsed_days              int64
project_value                  float64
type_of_work                       str
address                            str
project_description                str
permit_category                    str
applicant                          str
applicant_address                  str
property_use                       str
specific_use_category              str
building_contractor                str
building_contractor_address        str
issue_year                       int64
geo_local_area                     str
geom                               str
year_month                         str
latitude                       float64
longitude                      float64
dtype: object

Missing values:
permit_number                      0
permit_number_created_date         0
issue_date                    

,permit_number,permit_number_created_date,issue_date,permit_elapsed_days,project_value,type_of_work,address,project_description,permit_category,applicant,...,property_use,specific_use_category,building_contractor,building_contractor_address,issue_year,geo_local_area,geom,year_month,latitude,longitude
0,DB-2017-06475,2017-12-12,2018-05-11,150,15000.0,Demolition / Deconstruction,"3067 E 8TH AVENUE, Vancouver, BC V5M 1X4",Low Density Housing - Demolition / Deconstruct...,NaN,Kanwal Sekhon DBA: 88 Homes LTD.,...,Dwelling Uses,Single Detached House w/Sec Suite,PTL Contracting Ltd,"5649 ASH ST \r\nVancouver, BC V5Z 3G8",2018,Hastings-Sunrise,"{""coordinates"": [-123.0399664, 49.2631955], ""t...",2018-05,49.263196,-123.039966
1,BP-2017-06477,2017-12-12,2018-03-08,86,0.0,Salvage and Abatement,"3067 E 8TH AVENUE, Vancouver, BC V5M 1X4",Low Density Housing - Salvage and Abatement - ...,NaN,Kanwal Sekhon DBA: 88 Homes LTD.,...,Dwelling Uses,Single Detached House w/Sec Suite,PTL Contracting Ltd,"5649 ASH ST \r\nVancouver, BC V5Z 3G8",2018,Hastings-Sunrise,"{""coordinates"": [-123.0399664, 49.2631955], ""t...",2018-03,49.263196,-123.039966
2,BP-2017-06479,2017-12-12,2018-04-06,115,2500000.0,Addition / Alteration,"555 BURRARD STREET, Vancouver, BC",High Density Housing / Commercial - Addition /...,NaN,Laurie Schmidt DBA: Schmidt & Associates Deve...,...,Office Uses,General Office,NaN,NaN,2018,Downtown,"{""coordinates"": [-123.1189647, 49.2859917], ""t...",2018-04,49.285992,-123.118965
3,DB-2017-06483,2017-12-12,2018-07-16,216,911507.5,New Building,"3247 W 22ND AVENUE, Vancouver, BC V6L 1N1",Low Density Housing - New Building - To constr...,New Build - Low Density Housing,Maggie Tsai DBA: Formwerks Architectural Inc.,...,Dwelling Uses,Single Detached House,Gemlevy Projects Ltd,NaN,2018,Dunbar-Southlands,"{""coordinates"": [-123.1770113, 49.2530866], ""t...",2018-07,49.253087,-123.177011
4,BP-2017-06486,2017-12-12,2019-07-18,583,5996000.0,New Building,"397 W 7TH AVENUE, Vancouver, BC V1V 1V1",Certified Professional Program - New Building ...,NaN,Andrew Harmsworth DBA: GHL Consultants Ltd,...,Office Uses,General Office,NaN,NaN,2019,Mount Pleasant,"{""coordinates"": [-123.1126066, 49.2650645], ""t...",2019-07,49.265065,-123.112607


In [8]:
# Connect to SQLite database
conn = sqlite3.connect('building_permits.db')

# Write cleaned DataFrame to SQLite
df.to_sql(
    'permits',
    conn,
    if_exists='replace',
    index=False
)

print("Data successfully saved to SQLite.")

Data successfully saved to SQLite.


In [9]:
# QUERY 1: Where is construction activity concentrated?

query_1 = """
SELECT 
    geo_local_area,
    COUNT(*) AS total_permits,
    SUM(project_value) AS total_project_value,
    AVG(project_value) AS average_project_value
FROM permits
WHERE project_value > 0 
    AND project_value IS NOT NULL
    AND geo_local_area IS NOT NULL
GROUP BY geo_local_area
ORDER BY total_permits DESC;
"""

result = pd.read_sql_query(query_1, conn)

display(result)

,geo_local_area,total_permits,total_project_value,average_project_value
0,Downtown,7391,7.296323e+09,9.871902e+05
1,Kensington-Cedar Cottage,3151,1.710930e+09,5.429799e+05
2,Renfrew-Collingwood,2594,2.017210e+09,7.776447e+05
3,Hastings-Sunrise,2530,1.329616e+09,5.255397e+05
4,West End,2393,2.334887e+09,9.757154e+05
5,Kitsilano,2328,1.536389e+09,6.599609e+05
6,Sunset,2292,9.517670e+08,4.152561e+05
7,Fairview,2225,1.768380e+09,7.947774e+05
8,Dunbar-Southlands,2017,1.135860e+09,5.631431e+05
9,Riley Park,1929,1.622868e+09,8.413001e+05


In [10]:
# QUERY 2: Which specific property-use categories have the longest
# average processing times?

query_2 = """
WITH category_stats AS (
    SELECT
        specific_use_category,
        COUNT(*) AS total_permits,
        AVG(permit_elapsed_days) AS average_processing_days
    FROM permits
    WHERE permit_elapsed_days >= 0
        AND specific_use_category IS NOT NULL
    GROUP BY specific_use_category
)

SELECT 
    specific_use_category,
    total_permits,
    average_processing_days
FROM category_stats
WHERE total_permits >= 20
ORDER by average_processing_days DESC;
"""

result = pd.read_sql_query(query_2, conn)

display(result)

,specific_use_category,total_permits,average_processing_days
0,"Multiple Dwelling,Parking Garage",58,487.465517
1,"Secondary Suite,Duplex w/Secondary Suite",47,285.446809
2,"Multiple Dwelling,Retail Store",63,279.412698
3,"Duplex w/Secondary Suite,Secondary Suite",123,272.918699
4,"Duplex,Lock -off Unit",23,260.782609
...,...,...,...
80,Health Care Office,634,54.962145
81,Farmers Market,33,51.303030
82,General Office,5467,51.015548
83,Museum or Archives,62,31.887097


In [11]:
# QUERY 3: Do larger projects take longer to process?

query_3 = """
SELECT
    CASE
        WHEN project_value < 10000 THEN 'Small'
        WHEN project_value < 100000 THEN 'Medium'
        ELSE 'Large'
    END AS project_size,
    COUNT(*) AS total_permits,
    AVG(permit_elapsed_days) AS average_processing_days
FROM permits
WHERE project_value > 0
    AND permit_elapsed_days >= 0
GROUP BY project_size
ORDER BY average_processing_days DESC;
"""

result = pd.read_sql_query(query_3, conn)

display(result)

,project_size,total_permits,average_processing_days
0,Large,21395,172.928161
1,Medium,18853,115.622766
2,Small,3400,65.765882


In [12]:
# QUERY 4: How has construction activity changed over time?

query_4 = """
SELECT
    issue_year,
    COUNT(*) AS total_permits,
    SUM(project_value) AS total_project_value
FROM permits
WHERE project_value > 0
    AND issue_year IS NOT NULL
GROUP BY issue_year
ORDER BY issue_year DESC;
"""

result = pd.read_sql_query(query_4, conn)

display(result)

,issue_year,total_permits,total_project_value
0,2026,2691,3.912360e+09
1,2025,4381,5.910209e+09
2,2024,4229,4.385869e+09
3,2023,3945,5.969840e+09
4,2022,4738,4.045787e+09
5,2021,4025,2.468528e+09
6,2020,3713,4.823865e+09
7,2019,4818,3.387232e+09
8,2018,5705,3.139381e+09
9,2017,5403,2.752887e+09


In [13]:
# QUERY 5: Are certain geographic areas associated with longer permit
# processing times?

query_5 = """
SELECT
    geo_local_area,
    AVG(permit_elapsed_days) AS average_processing_time,
    COUNT(*) AS total_permits
FROM permits
WHERE permit_elapsed_days >= 0
    AND geo_local_area IS NOT NULL
GROUP BY geo_local_area
HAVING COUNT(*) >= 20
ORDER BY average_processing_time DESC;
"""

result = pd.read_sql_query(query_5, conn)

display(result)

,geo_local_area,average_processing_time,total_permits
0,South Cambie,178.515344,945
1,Oakridge,175.271830,1317
2,West Point Grey,168.042848,1587
3,Kerrisdale,167.506494,1386
4,Killarney,163.856136,1703
5,Sunset,163.217204,2790
6,Renfrew-Collingwood,162.268454,3319
7,Victoria-Fraserview,161.680841,2187
8,Dunbar-Southlands,160.778805,2595
9,Hastings-Sunrise,159.220787,3175


In [14]:
# QUERY 6: Which geographic areas had the highest construction 
# investment each year?

query_6 = """
WITH area_investment AS (
    SELECT
        issue_year,
        geo_local_area,
        SUM(project_value) AS total_investment
    FROM permits
    WHERE project_value > 0
        AND geo_local_area IS NOT NULL
    GROUP BY issue_year, geo_local_area
),

ranked_areas AS (
    SELECT 
        issue_year,
        geo_local_area,
        total_investment,
        RANK () OVER (
            PARTITION BY issue_year
            ORDER BY total_investment DESC
        ) AS investment_rank
    FROM area_investment
)

SELECT 
    issue_year,
    geo_local_area,
    total_investment
FROM ranked_areas
WHERE investment_rank = 1
ORDER BY issue_year DESC;
"""

result = pd.read_sql_query(query_6, conn)

display(result)

,issue_year,geo_local_area,total_investment
0,2026,Downtown,5.162952e+08
1,2025,Downtown,1.127686e+09
2,2024,Downtown,7.884241e+08
3,2023,Strathcona,1.799587e+09
4,2022,Downtown,6.320838e+08
5,2021,Oakridge,2.705783e+08
6,2020,Oakridge,1.542985e+09
7,2019,Downtown,1.126973e+09
8,2018,Downtown,6.607719e+08
9,2017,Downtown,4.481998e+08


In [15]:
conn.close()

In [16]:
# Create a cleaner dataset for Tableau which only contains relevant data
# for this analysis. Specifically, the 'project_description' column
# creates formatting issues in Tableau.

tableau_df = df[
  [
    'permit_number',
    'permit_number_created_date',
    'issue_date',
    'permit_elapsed_days',
    'project_value',
    'type_of_work',
    'specific_use_category',
    'property_use',
    'issue_year',
    'geo_local_area',
    'year_month',
    'latitude',
    'longitude'
    ]
].copy()

tableau_df.to_csv(
  "building_permits_tableau.csv",
  index=False
)